In [1]:
import ollama
import requests
import json
from ollama import chat
from ollama import ChatResponse

In [2]:
model = "llama3.2:1b"

#### Task 1: Interact with deployed LLM via python 


**Objective:**

Explore different techniques to interact with the deployed LLM.

**Task Description:**

1. Use Request libaray (HTTP Client) and send a POST request to interact with the LLM: [How To](https://requests.readthedocs.io/en/latest/user/quickstart/#make-a-request)

In [3]:
# Simple HTTP Request via requests

# Define the URL of the deployed LLM ( this port is forwarded from the docker container to the host system)
url = "http://localhost:11434/api/generate"

# Define the prompt as json
body_json = {
    "model": model,
    "prompt": "Describe Generative AI in two sentences."
}

# ADD HERE YOUR CODE
# HINT: Send the POST request using the json body
response = requests.post(url, json=body_json)

# Check if the request was successful
if response.status_code == 200:
    # Process the response
    response_text = response.text

    # Convert each line to json
    response_lines = response_text.splitlines()
    response_json = [json.loads(line) for line in response_lines]
    for line in response_json:
        # Print the response. No line break
        print(line["response"], end="")
else:
    print("Error:", response.status_code)


Generative AI refers to a type of artificial intelligence that can create new content, such as text, images, music, or videos, based on patterns and algorithms, allowing for the generation of diverse and novel outputs. This technology uses machine learning and natural language processing to learn from data and generate new content that can be used for various purposes, including creative projects, marketing, and research.

**Task Description:**

2. Use Ollama python library to interact with the LLM: [How To](https://pypi.org/project/ollama/)

- First use method ``ollama.chat(...)``
- First use method ``ollama.chat(...)`` with ``stream=True``

In [4]:
# API Call via ollama

# ADD HERE YOUR CODE

response: ChatResponse = chat(
    model = model, 
    messages = [
        {
            'role': 'user', 'content': 'Why is the sky blue?',
        }
    ]
)


print(response["message"]["content"])

The sky appears blue to the human eye because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century.

Here's what happens: when sunlight enters Earth's atmosphere, it consists of a spectrum of colors, including all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). The shorter wavelengths of light, such as blue and violet, are scattered more than the longer wavelengths, like red and orange, by the tiny molecules of gases in the atmosphere, such as nitrogen and oxygen.

This scattering effect is more pronounced when the sunlight is directly overhead, which is why the sky typically appears blue during the daytime. The blue light is scattered in all directions, giving the sky its blue color.

At sunrise and sunset, the light has to travel through more of the atmosphere to reach our eyes, which means that the shorter wavelengths of light are scattered away, leaving mainly t

In [5]:
# Streaming API Call via ollama

# Response streaming can be enabled by setting stream=True, 
# modifying function calls to return a Python generator where each part is an object in the stream.

# ADD HERE YOUR CODE
stream = chat(
  model = model, messages = [{
    'role': 'user', "content": "What is VfB Stuttgart?"
  }],
  stream = True
)


for chunk in stream:
  print(chunk["message"]["content"], end="", flush=True)

VfB Stuttgart, commonly referred to as VfB Stuttgart, is a German professional football club based in Stuttgart, Baden-Württemberg. The club was founded in 1894 and is one of the most successful and storied clubs in German football.

VfB Stuttgart plays in the Bundesliga, the top tier of German football, and has a significant following in the region. The club has a rich history, having won numerous domestic and international titles, including the German championship, the European Cup, and the UEFA Cup.

The club's name "VfB" comes from the German word for "football club," and "Stuttgart" refers to the city where the club is based. The team's kit is typically a blue and red color scheme, and they play their home games at the Neckarstadter Alte Arena, which has a capacity of over 50,000 spectators.

VfB Stuttgart is known for its passionate fan base, and the club has a strong following in the region, particularly in the southern part of Germany. The team has also had significant success 

#### Task 2: Experimenting with Prompt Techniques

**Objective:**

Objective: Explore different prompt techniques (Zero Shot, One Shot, and Few Shot) by sending different types of prompts to the LLM.

![image](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*QSpK--jqPiUU_OHuZvtUWA.png)

**Task Description:**

1. Create three prompts for a sentiment analysis task: a Zero Shot prompt, a One Shot prompt, and a Few Shot prompt. Use the examples from the table above.
2. Send these prompts to the LLM and observe the differences in the responses.
3. Compare and discuss the responses.

In [6]:
# ADD HERE YOUR PROMPTS

zero_shot_prompt = """Classify this Review. Answer with one word.: I loved this movie! Sentiment: """

one_shot_prompt = """Classify this review: I loved this movie! Sentiment: Positive. 
Classify this review. Answer with one word.: 

I don't like this chair. Sentiment: """

few_shot_prompt = """ Classify this review: I loved this movie! Sentiment: Positive. 
 Classify this review: I don't like this chair. Sentiment: Negative. 
 Classify this review. Answer with one word: who would use this product? Sentiment: """

# Stream the responses and print them
for idx, prompt in enumerate([zero_shot_prompt, one_shot_prompt, few_shot_prompt]):
    prompt_type = ["Zero-Shot", "One-Shot", "Few-Shot"][idx]
    print(f"\n--- {prompt_type} Prompt ---\n")
    print(f"User Prompt:\n{prompt}\n")
    
    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    print("Model Output:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print("\n-----------------------------\n")



--- Zero-Shot Prompt ---

User Prompt:
Classify this Review. Answer with one word.: I loved this movie! Sentiment: 

Model Output:
Positive
-----------------------------


--- One-Shot Prompt ---

User Prompt:
Classify this review: I loved this movie! Sentiment: Positive. 
Classify this review. Answer with one word.: 

I don't like this chair. Sentiment: 

Model Output:
Negative.

Sentiment: Negative.
-----------------------------


--- Few-Shot Prompt ---

User Prompt:
 Classify this review: I loved this movie! Sentiment: Positive. 
 Classify this review: I don't like this chair. Sentiment: Negative. 
 Classify this review. Answer with one word: who would use this product? Sentiment: 

Model Output:
Classification: 

1. I loved this movie! - Sentiment: Positive
2. I don't like this chair. - Sentiment: Negative
3. Who would use this product? - Furniture
-----------------------------



#### Task 3: Prompt Refinement and Optimization

**Objective:** 

Refine a prompt to improve the clarity and quality of the LLM's response.

**Task Description:**

- Start with a basic prompt asking the LLM to summarize a paragraph.
- Refine the prompt by adding specific instructions to improve the summary's quality. (Example: define how long the summary should be, define on which to focus in the summary)

In [7]:
# Original prompt
original_prompt = "Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."

# ADD HERE YOUR PROMPT
refined_prompt = "Summarize the following paragraph in one short sentence: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."

# Stream the responses and print them
for idx, prompt in enumerate([original_prompt, refined_prompt]):
    prompt_type = ["Original Prompt", "Refined Prompt"][idx]
    print(f"\n--- {prompt_type} ---\n")
    print(f"User Prompt:\n{prompt}\n")
    
    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    print("Model Output:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print("\n-----------------------------\n")



--- Original Prompt ---

User Prompt:
Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries.

Model Output:
The paragraph discusses Generative AI, a field of artificial intelligence focused on creating new content by analyzing patterns in existing data. This field can be applied to text, image, and music generation, and is being increasingly used in creative industries.
-----------------------------


--- Refined Prompt ---

User Prompt:
Summarize the following paragraph in one short sentence: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries.

Model Output:
Generative AI is a 

#### [Optional] Task 4: Structured Prompting with Roles (Pirate Theme)


**Objective:**

Learn how to use structured prompts that combine role assignment, clear instructions, and examples to improve the output of language models. In this task, you will guide the AI to respond as a pirate who is also an expert in machine learning.

**Instructions:**

- Role Assignment: In your prompt, specify the role of the AI as a Machine Learning Expert who speaks like a pirate.

- Instruction: Clearly state what you want the AI to explain or discuss in pirate language.

- Examples: Provide examples to guide the AI in using pirate lingo while explaining technical concepts.

In [8]:
# Combined Techniques Prompt with Pirate Theme

structured_prompt = "I want you to act like a pirate who is also an expert in machine learning. Answer my questions only in pirate language. My first question is for you to explain the Concept of generative AI like a pirate."

# Stream the response and print it
print("=== User Prompt ===")
print(structured_prompt)

stream = ollama.chat(
    model=model,
    messages=[{"role": "user", "content": structured_prompt}],
    stream=True,
)

print("\n=== Model Output ===")
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print("\n")


=== User Prompt ===
I want you to act like a pirate who is also an expert in machine learning. Answer my questions only in pirate language. My first question is for you to explain the Concept of generative AI like a pirate.

=== Model Output ===
Arrr, ye landlubber, ye want to know o' this "generative AI" business, eh? Alright then, listen close and I'll spin ye a yarn 'bout this fascinatin' contraption. 

Ye see, generative AI be a bit like creatin' yer own treasure map, matey. It's a way o' creatin' a whole new world o' possibilities with a wee bit o' code and some clever thinkin'. It's like buildin' a ship from scratch, but instead o' wood and rope, ye be usin' ones and zeros.

It takes in a piece o' knowledge, like a treasure map, and generates a whole new world o' possibilities, like a treasure chest overflowin' with booty. It's like havin' a magic box that can take in all sorts o' information, from words to images, and spit out a new, original creation, like a sea siren singin' a